# Task 1 — two label-free ideas, both negative

Branch `task1`. Standing submission scores **0.79** (physics + WiFo2 + MAE).

Selection rule for this branch, established in `notes/results.md`: changes made
**without consulting the 10 labels** transferred to the private set (MAE: +0.06
local -> +0.04 private, 1.4x inflation); changes **selected by** the labels did
not (physics: +0.68 local -> +0.03 private, 23x). Both ideas here are label-free,
so both were expected to transfer.

Neither worked. This notebook records why, so they are not retried.

| idea | verdict |
|---|---|
| windowed / oversampled delay transform | no gain — collapses to the same 0.857 |
| MAE on an augmented corpus | LOOCV 1.000 but **do not submit** — four signals say it is miscalibrated |

In [ ]:
import sys, warnings, json
from pathlib import Path

warnings.filterwarnings('ignore')
ROOT = Path.cwd() if (Path.cwd() / 'model.py').exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))

import numpy as np, torch, scipy.io as sio, h5py
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold, LeaveOneOut
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.metrics import f1_score, confusion_matrix

from physics_features import extract
import mae_pretrain as mae

SEEDS = [0, 1, 7, 13, 42, 99, 123, 2024, 31337, 555]
LOS = 1
W, FD = ROOT / 'weights/model_best.pkl', ROOT / 'weights/FastDepthV2_L1_Best.pth'

def load_mat(path, key):
    try:
        return sio.loadmat(path)[key]
    except NotImplementedError:
        with h5py.File(path, 'r') as f:
            a = f[key][...]
        if a.dtype.names == ('real', 'imag'):
            a = a['real'] + 1j * a['imag']
        return a.transpose(range(a.ndim)[::-1])

def to_x(Hc):
    return torch.stack([torch.tensor(Hc.real), torch.tensor(Hc.imag)], 1).float()

H    = load_mat(ROOT / 'dataset/Task1/H_train.mat', 'H')
y    = load_mat(ROOT / 'dataset/Task1/L_train.mat', 'label').ravel().astype(int)
H_te = load_mat(ROOT / 'dataset/Task1/H_test.mat', 'H')
x_tr, x_te = to_x(H), to_x(H_te)
base_corpus = torch.cat([x_tr, x_te])
N = len(y)
X_phys, _    = extract(H)
X_phys_te, _ = extract(H_te)
print(f'{N} labelled, {len(H_te)} unlabelled, prior {y.mean():.0%} LoS')

## Idea 1 — fix the delay transform

All the discriminative power is delay-domain: `first_tap_frac` (t=3.85) and
`rms_delay` (t=3.86). Every other feature family measured useless.

The current transform is a bare IFFT of 128 subcarriers:

```python
pdp = (np.abs(np.fft.ifft(h, axis=-1)) ** 2).mean((0, 1))
```

That is a **rectangular window** — -13 dB sidelobes. The hypothesis was that
sidelobe leakage from the dominant LoS tap smears energy across every delay bin,
depressing `first_tap_frac` and inflating `rms_delay`, compressing the class
contrast. Fixes: Hann/Hamming windowing (-31/-43 dB sidelobes) and zero-padding
for finer delay sampling. Pure DSP, no labels.

In [ ]:
def pdp_of(h, win='rect', nfft=128):
    K = h.shape[-1]
    w = {'rect': np.ones(K), 'hann': np.hanning(K),
         'hamming': np.hamming(K), 'blackman': np.blackman(K)}[win]
    return (np.abs(np.fft.ifft(h * w, n=nfft, axis=-1)) ** 2).mean((0, 1))

def delay_feats(h, win, nfft):
    pdp = pdp_of(h, win, nfft); os_ = nfft // h.shape[-1]
    p = pdp / pdp.sum(); tau = np.arange(len(p)); mt = (tau * p).sum()
    return {'first_tap_frac': p[:os_].sum(), 'peak_frac': p.max(),
            'rms_delay': np.sqrt((((tau - mt) ** 2) * p).sum()) / os_,
            'pdp_entropy': -(p * np.log(p + 1e-12)).sum()}

def tstat(v):
    a, b = v[y == 0], v[y == 1]
    return abs(a.mean() - b.mean()) / (np.sqrt(a.var()/len(a) + b.var()/len(b)) + 1e-12)

names = ['first_tap_frac', 'peak_frac', 'rms_delay', 'pdp_entropy']
print(f"{'window':10s}{'nfft':>6}" + ''.join(f'{n:>17s}' for n in names))
for win in ('rect', 'hann', 'hamming', 'blackman'):
    for nfft in (128, 512):
        F = {n: np.array([delay_feats(H[i], win, nfft)[n] for i in range(N)]) for n in names}
        tag = '  <- current' if (win == 'rect' and nfft == 128) else ''
        print(f'{win:10s}{nfft:>6}' + ''.join(f'{tstat(F[n]):>17.2f}' for n in names) + tag)

### The hypothesis was wrong

`first_tap_frac` gets **worse** with windowing (3.85 -> 3.36 Hann) and much worse
with zero-padding (1.87 at nfft=512). The dominant effect is not sidelobe leakage
but **main-lobe widening**: a window spreads the direct path's energy into
neighbouring delay bins, so less of it lands in tap 0. Oversampling splits it
further.

`rms_delay` moves the opposite way (3.86 -> 4.72 at Hamming/512) — it is a
second-moment statistic and benefits from finer sampling and cleaner tails.

The two features want opposite transforms, and **choosing between them by t-stat
would be reading the labels** — the selection leakage that cost 0.08 in
`task1_physics.ipynb`. So all four settings go into the pool and `SelectKBest`
picks inside each fold.

In [ ]:
def delay_block(Hs, win, nfft):
    return np.array([[v for v in delay_feats(h, win, nfft).values()] for h in Hs])

CFG = [('hann', 128), ('hamming', 128), ('rect', 512), ('hamming', 512)]
D    = np.hstack([delay_block(H,    w, n) for w, n in CFG])
D_te = np.hstack([delay_block(H_te, w, n) for w, n in CFG])

m, _ = mae.build(W, FD); mae.pretrain(m, base_corpus, epochs=20, seed=0)
V, V_te = mae.features(m, x_tr), mae.features(m, x_te)

def evaluate(X, k=3):
    pipe = lambda: make_pipeline(StandardScaler(), SelectKBest(f_classif, k=min(k, X.shape[1])),
                                 LogisticRegression(C=1, max_iter=3000))
    sc = []
    for s in SEEDS:
        oof = np.zeros(N, int)
        for itr, iva in StratifiedKFold(5, shuffle=True, random_state=s).split(X, y):
            p = pipe(); p.fit(X[itr], y[itr]); oof[iva] = p.predict(X[iva])
        sc.append(f1_score(y, oof, pos_label=LOS, zero_division=0))
    lo = np.zeros(N, int)
    for itr, iva in LeaveOneOut().split(X):
        p = pipe(); p.fit(X[itr], y[itr]); lo[iva] = p.predict(X[iva])
    return np.mean(sc), np.std(sc), f1_score(y, lo, pos_label=LOS, zero_division=0)

print(f"{'feature set':38s}{'k':>3}{'binF1':>8}{'std':>7}{'LOO':>7}")
for nm, X in [('physics + wifo+MAE (current best)', np.hstack([X_phys, V])),
              ('  + windowed delay',                np.hstack([X_phys, D, V])),
              ('windowed delay + wifo only',        np.hstack([D, V])),
              ('physics + windowed, no wifo',       np.hstack([X_phys, D]))]:
    for k in (3, 5):
        r = evaluate(X, k)
        print(f'{nm:38s}{k:>3}{r[0]:>8.3f}{r[1]:>7.3f}{r[2]:>7.3f}')

### Verdict: no gain

Everything lands on **0.857 with std 0.000** — the same value the 54-member
ensemble produced, and the same LOOCV value every model since `mean-pool +
logreg` has given.

That number is `TP3 FP0 FN1`: catch three of the four LoS samples, no false
positives. It is the stable answer this dataset supports. The 0.914 configuration
catches the fourth LoS sample in some fold splits and that is the *entire*
difference between the two numbers.

**One sample separates 0.857 from 0.914.** Every method that does not happen to
catch it converges to the same score. That is the resolution floor of a 10-sample
evaluation.

## Idea 2 — MAE on an augmented corpus

Augmentation failed for the *classifier* (recorded in `notes/results.md`). MAE is
a different use: self-supervised, no labels to overfit, and the corpus is only 30
samples. Phase rotation, amplitude scaling, antenna permutation and AWGN are all
label-preserving channel nuisances.

In [ ]:
def augment(x, g):
    z = x[0] + 1j * x[1]
    z = z * np.exp(1j * g.uniform(0, 2*np.pi))     # arbitrary phase reference
    z = z * g.uniform(0.5, 2.0)                    # path loss
    z = z[:, g.permutation(z.shape[1]), :]         # antenna order
    p = np.mean(np.abs(z) ** 2)
    s = np.sqrt(p * 10 ** (-g.uniform(20, 35)/10) / 2)
    z = z + s * (g.normal(size=z.shape) + 1j * g.normal(size=z.shape))
    return torch.tensor(np.stack([z.real, z.imag]), dtype=torch.float32)

def make_corpus(mult):
    if mult == 0:
        return base_corpus
    out = [base_corpus]
    for r in range(mult):
        g = np.random.default_rng(500 + r)
        out.append(torch.stack([augment(base_corpus[i].numpy(), g) for i in range(len(base_corpus))]))
    return torch.cat(out)

print(f"{'config':24s}{'mseed':>7}{'recon':>9}{'binF1':>8}{'std':>7}{'LOO':>7}   LOO confusion")
for mult, eps in [(0, 20), (9, 5), (4, 10), (9, 3)]:
    c = make_corpus(mult)
    for sd in (0, 1, 2):
        mm, _ = mae.build(W, FD); h = mae.pretrain(mm, c, epochs=eps, seed=sd)
        X = np.hstack([X_phys, mae.features(mm, x_tr)])
        a, b, lo = evaluate(X)
        oof = np.zeros(N, int)
        for itr, iva in LeaveOneOut().split(X):
            p = make_pipeline(StandardScaler(), SelectKBest(f_classif, k=3),
                              LogisticRegression(C=1, max_iter=3000))
            p.fit(X[itr], y[itr]); oof[iva] = p.predict(X[iva])
        (tn, fp), (fn, tp) = confusion_matrix(y, oof, labels=[0, 1])
        tag = 'no augmentation' if mult == 0 else f'30+{mult}x aug, {eps}ep'
        print(f'{tag:24s}{sd:>7}{h[-1]:>9.5f}{a:>8.3f}{b:>7.3f}{lo:>7.3f}   TN{tn} FP{fp} FN{fn} TP{tp}')
        if mult == 0: break
    print()

### LOOCV hits 1.000 — and that is the trap

`30+9x aug, 5ep` gives **LOOCV 1.000 (TP4 FP0 FN0) on every MAE seed**. It is the
first model on this branch to catch all four LoS samples, breaking the `FN=1`
barrier that ensembling, windowing, extra features and epoch tuning all failed to
move. `3ep` is not seed-stable, so this is not merely a lucky init.

That looked like the strongest candidate yet. It is not. Check the test-set
probabilities against the model that actually scored 0.79.

In [ ]:
def fit_predict(mult, eps):
    c = make_corpus(mult)
    mm, _ = mae.build(W, FD); mae.pretrain(mm, c, epochs=eps, seed=0)
    X    = np.hstack([X_phys,    mae.features(mm, x_tr)])
    X_t  = np.hstack([X_phys_te, mae.features(mm, x_te)])
    p = make_pipeline(StandardScaler(), SelectKBest(f_classif, k=3),
                      LogisticRegression(C=1, max_iter=3000)).fit(X, y)
    return p.predict_proba(X_t)[:, list(p.classes_).index(LOS)]

q_base, q_aug = fit_predict(0, 20), fit_predict(9, 5)

print(f"{'sample':>7}{'no aug':>10}{'9x aug':>10}{'shift':>9}")
for i in range(len(q_base)):
    flag = '  <- reversed' if abs(q_aug[i]-q_base[i]) > 0.4 else ''
    print(f'{i:>7}{q_base[i]:>10.3f}{q_aug[i]:>10.3f}{q_aug[i]-q_base[i]:>+9.3f}{flag}')

for tag, q in [('no aug (the 0.79 model)', q_base), ('9x augmented', q_aug)]:
    p = (q >= .5).astype(int)
    print(f'\n{tag}: balance {np.bincount(p, minlength=2)}  LoS rate {p.mean():.0%}'
          f'  mean P {q.mean():.3f}')
print(f'\ntraining prior: {y.mean():.0%} LoS')

### Verdict: do not submit

Four independent signals, all pointing the same way:

| signal | reading |
|---|---|
| test LoS rate **25%** vs 40% prior | under-predicts the positive class, and recall is already the bottleneck |
| confident predictions **reversed** (sample 2: 0.978 -> 0.143) | disagrees wholesale with the model that scored 0.79 |
| MAE recon loss **0.75 -> 1.29** | the encoder is fitting augmented statistics the real test samples do not share |
| 5-fold **0.914 -> 0.886** | the protocol with *less* training data got worse |

LOOCV 1.000 is the only signal pointing the other way, and it is the one most
likely to saturate: LOOCV on 10 samples is **10 binary outcomes**, and 10/10 is
well within reach of a model that is badly calibrated on anything new.

Mean P(LoS) collapses 0.501 -> 0.301. That is not a refinement, it is a different
model — and its disagreements are concentrated exactly where the 0.79 submission
was most confident.

## Summary

Both ideas were label-free, so both were expected to transfer under this branch's
own rule. Neither delivered:

- **windowed delay transform**: hypothesis wrong (main-lobe widening dominates
  sidelobe leakage), no CV gain, everything collapses to the stable 0.857
- **MAE on augmented corpus**: LOOCV 1.000 but four signals say miscalibrated;
  predictions saved to `experiments/task1_mae_aug/` but **not recommended**

The standing 0.79 submission remains the best candidate. Local evidence is
exhausted: one LoS sample separates every model on this branch, which is below
what a 10-sample evaluation can resolve.